# Sequence architecture patterns

**Learning objective:** Map many-to-one and many-to-many tasks to TensorFlow model shapes.

This notebook is part of the TensorFlow/Keras learning track. It is designed to be read top-to-bottom: intuition → shapes → mathematics → TensorFlow implementation → observed result → interpretation.

> GitHub renders the committed executed output as a static learning artifact. Clone the repository and rerun it in Jupyter/VS Code for live experimentation.


In [1]:
import os, warnings, random
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "bundled with TensorFlow")
print("Execution device(s):", [d.device_type for d in tf.config.list_logical_devices()])


2026-09-21 07:13:46.051652: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789974826.066361    3107 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789974826.070602    3107 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.1
Keras: 3.15.1
Execution device(s): ['CPU']


2026-09-21 07:13:47.871098: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
inp=tf.keras.Input((7,4))
many_to_one=tf.keras.Model(inp, tf.keras.layers.Dense(2)(tf.keras.layers.GRU(6)(inp)), name="many_to_one")
inp2=tf.keras.Input((7,4))
many_to_many=tf.keras.Model(inp2, tf.keras.layers.Dense(2)(tf.keras.layers.GRU(6,return_sequences=True)(inp2)), name="many_to_many")
x=tf.zeros((3,7,4))
print("many→one:",many_to_one(x).shape); print("many→many:",many_to_many(x).shape)


many→one: (3, 2)
many→many: (3, 7, 2)


In [3]:
patterns=pd.DataFrame({"pattern":["many→one","many→many (aligned)","encoder→decoder"],"example":["sentiment / forecast","tag every timestep","translation / generation"],"typical RNN output":["final state","return_sequences=True","state passed to decoder"]})
display(patterns)


,pattern,example,typical RNN output
0,many→one,sentiment / forecast,final state
1,many→many (aligned),tag every timestep,return_sequences=True
2,encoder→decoder,translation / generation,state passed to decoder


Architecture follows the output contract. Always write the desired input and output tensor shapes before selecting recurrent-layer options.
